<a href="https://colab.research.google.com/github/hugoalonso1704/api-analisis-monedas/blob/main/api_tarea_graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

url = "https://api.frankfurter.dev/v1/latest"
r = requests.get(url, params={"from": "EUR"})
r.raise_for_status()
data = r.json()

df = pd.DataFrame(list(data["rates"].items()), columns=["moneda", "tasa"])
df["base"] = data["base"]
df["fecha"] = data["date"]
print(df.head())

df.to_csv("tipos_cambio_eur.csv", index=False)
from google.colab import files
files.download("tipos_cambio_eur.csv")

  moneda    tasa base       fecha
0    AUD  1.6176  EUR  2026-10-02
1    BRL  5.8610  EUR  2026-10-02
2    CAD  1.5984  EUR  2026-10-02
3    CHF  0.9279  EUR  2026-10-02
4    CNY  7.5259  EUR  2026-10-02


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import requests
import pandas as pd
from datetime import date, timedelta
from google.colab import files

inicio = (date.today() - timedelta(days=365)).isoformat()
r = requests.get(f"https://api.frankfurter.dev/v1/{inicio}..", params={"from": "EUR"})
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data["rates"]).T
df.index = pd.to_datetime(df.index)
df.index.name = "fecha"
df = df.sort_index()
df.columns.name = "moneda"
stats = df.describe().T
stats["mediana"] = df.median()
stats["varianza"] = df.var()
stats["rango"] = df.max() - df.min()
stats["asimetria"] = df.skew()
stats["curtosis"] = df.kurt()
stats["coef_variacion_%"] = df.std() / df.mean() * 100
stats["variacion_%"] = (df.iloc[-1] / df.iloc[0] - 1) * 100
stats = stats.round(4)

print(stats)

stats.to_csv("estadisticos_generales.csv")
files.download("estadisticos_generales.csv")

        count        mean       std         min         25%         50%  \
moneda                                                                    
AUD     255.0      1.6799    0.0608      1.6095      1.6306      1.6501   
BGN      62.0      1.9558    0.0000      1.9558      1.9558      1.9558   
BRL     255.0      6.0485    0.1907      5.7449      5.8847      6.0126   
CAD     255.0      1.6119    0.0123      1.5680      1.6064      1.6130   
CHF     255.0      0.9254    0.0101      0.9008      0.9175      0.9250   
CNY     255.0      7.9967    0.2072      7.5259      7.7994      7.9916   
CZK     255.0     24.2822    0.0925     24.0830     24.2100     24.2770   
DKK     255.0      7.4721    0.0028      7.4656      7.4694      7.4725   
GBP     255.0      0.8662    0.0079      0.8487      0.8596      0.8664   
HKD     255.0      9.0783    0.0986      8.8084      9.0043      9.0763   
HUF     255.0    373.1563   13.0535    349.5300    362.2300    376.6300   
IDR     255.0  20027.7424

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!pip install -q xlsxwriter

In [9]:
import requests
import pandas as pd
from datetime import date, timedelta
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import DataPoint
from google.colab import files

inicio = (date.today() - timedelta(days=365)).isoformat()
r = requests.get(f"https://api.frankfurter.dev/v1/{inicio}..", params={"from": "EUR"})
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data["rates"]).T
df.index = pd.to_datetime(df.index)
df.index.name = "fecha"
df = df.sort_index().astype(float)
df.columns.name = "moneda"

stats = df.describe().T
stats["mediana"] = df.median()
stats["varianza"] = df.var()
stats["rango"] = df.max() - df.min()
stats["asimetria"] = df.skew()
stats["curtosis"] = df.kurt()
stats["coef_variacion_%"] = df.std() / df.mean() * 100
stats["variacion_%"] = (df.iloc[-1] / df.iloc[0] - 1) * 100
stats = stats.round(4).sort_values("variacion_%")
stats.index.name = "moneda"

n = len(stats)
m = len(df)

ppal = [x for x in ["USD", "GBP", "CHF", "JPY"] if x in stats.index]
resumen = stats.loc[ppal, ["min", "mean", "max"]]
k = len(resumen)

def col_excel(tabla, nombre):
    return list(tabla.columns).index(nombre) + 2
archivo = "analisis_divisas.xlsx"
with pd.ExcelWriter(archivo, engine="openpyxl") as writer:
    stats.to_excel(writer, sheet_name="estadisticos")
    df.to_excel(writer, sheet_name="datos")
    resumen.to_excel(writer, sheet_name="resumen")
    wb = writer.book
    ws = wb.create_sheet("graficos")
    ws_est, ws_dat, ws_res = wb["estadisticos"], wb["datos"], wb["resumen"]

    c1 = BarChart()
    c1.type = "bar"
    c1.title = "Variación % frente al EUR (último año)"
    j = col_excel(stats, "variacion_%")
    c1.add_data(Reference(ws_est, min_col=j, min_row=1, max_row=n + 1), titles_from_data=True)
    c1.set_categories(Reference(ws_est, min_col=1, min_row=2, max_row=n + 1))
    s = c1.series[0]
    s.invertIfNegative = False
    for i, v in enumerate(stats["variacion_%"]):
        pt = DataPoint(idx=i)
        pt.graphicalProperties.solidFill = "FF0000" if v < 0 else "00A050"
        s.dPt.append(pt)
    c1.legend = None
    c1.x_axis.delete = False
    c1.y_axis.delete = False
    c1.width, c1.height = 18, 20
    ws.add_chart(c1, "A1")

    c2 = BarChart()
    c2.type = "col"
    c2.title = "Volatilidad (coef. de variación %)"
    j = col_excel(stats, "coef_variacion_%")
    c2.add_data(Reference(ws_est, min_col=j, min_row=1, max_row=n + 1), titles_from_data=True)
    c2.set_categories(Reference(ws_est, min_col=1, min_row=2, max_row=n + 1))
    c2.legend = None
    c2.x_axis.delete = False
    c2.y_axis.delete = False
    c2.width, c2.height = 24, 10
    ws.add_chart(c2, "L1")

    c3 = LineChart()
    c3.title = "Evolución EUR/USD y EUR/GBP"
    cats = Reference(ws_dat, min_col=1, min_row=2, max_row=m + 1)
    if "USD" in df.columns:
        ju = df.columns.get_loc("USD") + 2
        c3.add_data(Reference(ws_dat, min_col=ju, min_row=1, max_row=m + 1), titles_from_data=True)
        c3.set_categories(cats)
    c3.x_axis.number_format = "mmm-yy"
    c3.x_axis.delete = False
    c3.y_axis.delete = False
    if "GBP" in df.columns:
        c3b = LineChart()
        jg = df.columns.get_loc("GBP") + 2
        c3b.add_data(Reference(ws_dat, min_col=jg, min_row=1, max_row=m + 1), titles_from_data=True)
        c3b.y_axis.axId = 200
        c3b.y_axis.crosses = "max"
        c3b.y_axis.delete = False
        c3 += c3b
    c3.width, c3.height = 24, 10
    ws.add_chart(c3, "L22")

    c4 = BarChart()
    c4.type = "col"
    c4.title = "Mín / Media / Máx (principales)"
    c4.add_data(Reference(ws_res, min_col=2, max_col=4, min_row=1, max_row=k + 1), titles_from_data=True)
    c4.set_categories(Reference(ws_res, min_col=1, min_row=2, max_row=k + 1))
    c4.x_axis.delete = False
    c4.y_axis.delete = False
    c4.width, c4.height = 24, 10
    ws.add_chart(c4, "L43")

files.download(archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>